In [ ]:
# pylint: disable=wrong-import-position
# pylint: disable=wrong-import-order
# pylint: disable=invalid-name
# pylint: disable=ungrouped-imports
# pylint: disable=redefined-outer-name


"""Demo smart home agent built with polycog cognition."""

## critical for running the tutorial on jupyter notebook
## ignore if running on terminal
import nest_asyncio2  # type: ignore[import-untyped]

nest_asyncio2.apply()

# Hierarchical Decision Processes

In previous tutorials, we built flat decision processes that managed simple interaction loops (e.g., controlling a single lamp via a chat window). However, as smart home environments expand to include dozens of devices across multiple rooms, placing all intent parsing and decision logic into a single flat process becomes fragile and unwieldy.

In this tutorial, we will construct **hierarchical decision processes** that decompose complex multi-step tasks into modular sub-processes with dedicated sub-states.

## Prerequisites: Start the Simulation Environment

Before initializing and running your agent, ensure the smart home simulation server is running in a separate terminal using **realistic mode**:

```bash
python smart_home/smart_home.py --mode realistic
```

### Environment Overview: Multi-Room & Multi-Device Setup

Unlike earlier single-device examples, running the simulation in `--mode realistic` introduces a multi-room smart home with diverse state capabilities. Before defining our data contracts and operators, let's take a moment to review the available devices and their operational states across the home:

- **Lighting (All Rooms):** The **bedroom**, **living room**, **kitchen**, **bathroom**, and **laundry room** each feature a light device that can be set to `ON` or `OFF`.
- **Kitchen:** Contains a **dishwasher** with states `IDLE` or `RUNNING`.
- **Laundry Room:**
  - **Washer** & **Dryer**: Operational states can be `IDLE` or `RUNNING`.
  - **Laundry Basket**: Sensor state can be `EMPTY` or `FULL`.
- **Living Room:** Features a **main door lock** that can be `LOCKED` or `UNLOCKED`.

This expanded device surface area is why a simple flat state is no longer sufficient, requiring the multi-device `Observation` models and structured intent hierarchies we will build in the following steps.

### Try It Out: Inspecting Environment Telemetry

Before running your agent code, take a look at the terminal window where `smart_home.py --mode realistic` is executing:

- **Watch the Live Observations:** Notice the structured observation payload generated by the simulation in real time. This dynamic telemetry stream reflects the exact status of every room and device in the home.
- **Manually Toggle Device States:** Use the simulation TUI to manually flip a few switches—turn the bedroom light **ON**, lock the main door, or set the laundry basket to **FULL**.

Observe how the generated state payload instantly updates to reflect your manual changes. This raw telemetry stream is what our `SmartHomeDevices` sensor will ingest and parse into typed `Observation` contracts during the **Perceive** phase of the agent loop!

--- 
## Step 1: Define Expanded Environment Data Contracts (`SmartHomeDevices`)

To manage a complex multi-device environment, we expand our environment data contracts beyond a single device. We define:
- `Devices`: Multi-device taxonomy mapping keys to human-readable descriptions of devices in various rooms.
- `DeviceState` & `ControlSignal`: Operating states and command signals.
- `Observation`: Strongly typed Pydantic payload representing simultaneous telemetry snapshots across all home devices.
- `Action`: Command payload pairing a specific target `Devices` enum with a `ControlSignal` that actuates it.

In [ ]:
import os
from dataclasses import dataclass
from enum import StrEnum
from typing import Any, Literal, cast

from cognition import (
    Actuator,
    AutoDocEnum,
    Cogent,
    DecisionProcess,
    DocEnum,
    IOContainer,
    Operator,
    Sensor,
)
from cognition.language import EnumClassifier
from pydantic import BaseModel
from pydantic_ai.models import infer_model

from smart_home.client import SmartHomeClient
from utils import run_cogent

assert "OPENAI_API_KEY" in os.environ, "Environment variable OPENAI_API_KEY is not set."
llm_model = infer_model("openai:gpt-4o")

# =============================================================================
# 1. Environment Data Contracts
# =============================================================================


class Devices(DocEnum):
    """Smart home device taxonomy."""

    BEDROOM = "bedroom", "light device in the bedroom"
    LIVING_ROOM = "living_room", "light device in the living room"
    KITCHEN = "kitchen", "light device in the kitchen"
    BATHROOM = "bathroom", "light device in the bathroom"
    MAIN_DOOR = "main_door", "main door lock device in living room"
    DISHWASHER = "dishwasher", "dishwasher device in kitchen"
    WASHER = "washer", "washing machine, washer in laundry room"
    DRYER = "dryer", "drying machine, dryer in laundry room"
    LAUNDRY_BASKET = "laundry_basket", "laundry basket device in laundry room"


LIGHTS = {
    Devices.BATHROOM,
    Devices.BEDROOM,
    Devices.KITCHEN,
    Devices.LIVING_ROOM,
    Devices.LAUNDRY_BASKET,
}


class DeviceState(StrEnum):
    """Smart home device operational states."""

    ON = "on"
    OFF = "off"
    LOCKED = "locked"
    UNLOCKED = "unlocked"
    IDLE = "idle"
    RUNNING = "running"
    EMPTY = "empty"
    FULL = "full"


class ControlSignal(StrEnum):
    """Control signals for smart home devices."""

    TURN_ON = "turn_on"
    TURN_OFF = "turn_off"
    START = "start"
    STOP = "stop"
    FILL = "fill"
    EMPTY = "empty"
    LOCK = "lock"
    UNLOCK = "unlock"


class Observation(BaseModel, frozen=True):
    """State observation payload across all environment devices."""

    bedroom: Literal[DeviceState.ON, DeviceState.OFF]
    living_room: Literal[DeviceState.ON, DeviceState.OFF]
    kitchen: Literal[DeviceState.ON, DeviceState.OFF]
    bathroom: Literal[DeviceState.ON, DeviceState.OFF]
    main_door: Literal[DeviceState.LOCKED, DeviceState.UNLOCKED]
    dishwasher: Literal[DeviceState.IDLE, DeviceState.RUNNING]
    washer: Literal[DeviceState.IDLE, DeviceState.RUNNING]
    dryer: Literal[DeviceState.IDLE, DeviceState.RUNNING]
    laundry_basket: Literal[DeviceState.EMPTY, DeviceState.FULL]


@dataclass(frozen=True)
class Action:
    """Command action payload targeting a specific device."""

    device: Devices
    signal: ControlSignal


@dataclass(frozen=True)
class Utterance:
    """Natural language chat message contract."""

    phrase: str | None

## Step 2: Define Structured Intent Contracts (`StructuredIntent`)

Parsing natural language in a complex environment requires extracting both the **action type** and the **target entity**. Rather than creating an exhaustive list of single enums (e.g., `TURN_ON_BEDROOM`, `TURN_ON_KITCHEN`), we define a composite data contract called `StructuredIntent`.

- `ResidentIntent`: Broad categories of user requests.
- `StructuredIntent`: Sub-state model pairing `ResidentIntent` with a target object (`Devices` or `Mode`).

In addition, we include another enum `KState` that captures a knowledge state that indicates that the cogent doesn't know how to classify an utterance to one of the known categories.

In [12]:
class ResidentIntent(DocEnum):
    """Supported resident intent categories."""

    TURN_OFF = (
        "turn_off",
        "user is asking the assistant to turn a device off or stop a device",
    )
    TURN_ON = (
        "turn_on",
        "user is asking the assistant to turn a device on or start a device",
    )
    LOCK = "lock", "user is asking the assistant to lock a device"
    UNLOCK = "unlock", "user is asking the assistant to unlock a device"
    SET_MODE = "set_mode", "user is asking the assistant to set an operational mode"


class MyIntent(StrEnum):
    """Standard conversational responses from the assistant."""

    DONTKNOW = "I am sorry; I am not programmed to respond to that."
    CONFIRMATION = "Done."


class Mode(AutoDocEnum):
    """Operating modes for the home assistant."""

    DAY = "day mode of smart assistant"
    EVENING = "evening mode of smart assistant"
    MIDNIGHT = "midnight mode of smart assistant"


class KState(AutoDocEnum):
    """Enum representing knowledge states"""

    UNKNOWN = "symbol representing that something is unknown"


@dataclass
class StructuredIntent:
    """Sub-state storing parsed intent category and target entity."""

    intent: ResidentIntent | KState | None = None
    object: Mode | Devices | KState | None = None

## Step 3: Implement the Decision Sub-Process
A decision sub-process in `cognition` operates much like a top-level decision process: it maintains its own dedicated state, set of operators, and termination criteria. The key difference is lifecycle management - a sub-process is instantiated and executed directly inside a parent operator. In our cogent, resolving a `StructuredIntent` occurs across two sequential stages within this sub-process: category extraction (`ExtractIntent`) first identifies the broad action verb (e.g., `ResidentIntent.TURN_ON`), followed by entity extraction (`ExtractIntentDevice` or `ExtractIntentMode`), which identifies the target noun (e.g., `Devices.KITCHEN` or `Mode.MIDNIGHT`). Both operators execute within the sub-process lifecycle and operate exclusively on the isolated `StructuredIntent` sub-state.

### Sub-Process Operator Implementation
1. `ExtractIntent`
The `ExtractIntent` operator performs the initial classification pass. It uses an `EnumClassifier` tuned to `ResidentIntent` to determine the user's high-level request. If classification fails, it safely falls back to `UNKNOWN` to avoid downstream errors.

2. `ExtractIntentDevice`
The `ExtractIntentDevice` operator executes when the intent requires a physical hardware target (`TURN_ON` or `TURN_OFF`). It runs an `EnumClassifier` over the `Devices` enum to identify the target device.

3. `ExtractIntentMode`
The `ExtractIntentMode` operator executes when the resident requests an operational state change (`SET_MODE`). It uses an `EnumClassifier` over the `Mode` enum to resolve the target system setting (e.g., `DAY`, `EVENING`, or `MIDNIGHT`).

4. `is_process_human_intent_terminal`
The termination predicate function checks whether both the intent verb and object noun have been resolved, signaling to `cognition` that the child sub-process is complete and can return control to the parent process.

In [ ]:
class ExtractIntent(Operator[StructuredIntent]):
    """
    Super-process operator: ProcessHumanIntent
    INIT: Instantiate an EnumClassifer for ResidentIntent
    WHEN: Intent has not been unclassified
    THEN: Classify user utterance into a ResidentIntent, KState.UNKNOWN if failed
    """

    def __init__(self, name: str, **kwargs: Any) -> None:
        super().__init__(name, **kwargs)
        self._interpreter = EnumClassifier(
            enum_type=ResidentIntent,
            task_desc="what is the resident asking the assistant to do",
        )

    def can_perform(self, state: StructuredIntent, io: IOContainer) -> bool:
        return state.intent is None

    def perform(self, state: StructuredIntent, io: IOContainer) -> None:
        utterance = cast(
            Utterance, io.a.utterance
        )  ## read the utterace supplied by super-operator
        intent = self._interpreter(utterance.phrase, llm=llm_model, num_trials=1)[0]
        state.intent = intent if intent else KState.UNKNOWN
        if state.intent is KState.UNKNOWN:
            state.object = KState.UNKNOWN
        print(
            f"--> [Process intent decision process] -> recognized intent as {state.intent}"
        )


class ExtractIntentDevice(Operator[StructuredIntent]):
    """
    Super-process operator: ProcessHumanIntent
    INIT: Instantiate an EnumClassifer for Device
    WHEN: Intent is ResidentIntent.TURN_OFF/TURN_ON and Object is None
    THEN: Classify user utterance into a Device, KState.UNKNOWN if failed
    """

    def __init__(self, name: str, **kwargs: Any) -> None:
        super().__init__(name, **kwargs)
        self._interpreter = EnumClassifier(
            enum_type=Devices, task_desc="which device is the resident referring to"
        )

    def can_perform(self, state: StructuredIntent, io: IOContainer) -> bool:
        return bool(
            state.intent in (ResidentIntent.TURN_OFF, ResidentIntent.TURN_ON)
            and not state.object
        )

    def perform(self, state: StructuredIntent, io: IOContainer) -> None:
        utterance = cast(Utterance, io.a.utterance)
        assert utterance.phrase, "expect parent operator to fill value"
        device = self._interpreter(utterance.phrase, llm_model, num_trials=1)[0]
        state.object = device if device else KState.UNKNOWN
        print(
            f"--> [Process intent decision process] -> recognized devices {state.object}"
        )


class ExtractIntentMode(Operator[StructuredIntent]):
    """
    Super-process operator: ProcessHumanIntent
    INIT: Instantiate an EnumClassifer for Mode
    WHEN: Intent is ResidentIntent.TURN_OFF/TURN_ON and Object is None
    THEN: Classify user utterance into a Mode, unknown if failed
    """

    def __init__(self, name: str, **kwargs: Any) -> None:
        super().__init__(name, **kwargs)
        self._mode_interpreter = EnumClassifier(
            enum_type=Mode,
            task_desc="which mode is the user wanting to set",
        )

    def can_perform(self, state: StructuredIntent, io: IOContainer) -> bool:
        return bool(state.intent is ResidentIntent.SET_MODE and not state.object)

    def perform(self, state: Any, io: IOContainer) -> None:
        utterance = cast(Utterance, io.a.utterance)
        assert utterance.phrase, "expect parent operator to fill value"
        mode = self._mode_interpreter(utterance.phrase, llm_model, num_trials=1)[0]
        state.object = mode if mode else KState.UNKNOWN
        print(
            f"--> [Process intent decision process] -> recognized mode as {state.object}"
        )


def is_process_human_intent_terminal(state: StructuredIntent, _io: IOContainer) -> bool:
    """Super-process operator: ProcessHumanIntent
    terminate decision process when both intent and object have been classified"""
    return state.intent is not None and state.object is not None

## Step 4: Embed the Decision Sub-Process Inside an Operator
Now we build the main decision process and integrate our decision sub-process an operator.

Instead of handling raw natural language interpretation directly within the main decision process, the operator `ProcessHumanIntent` encapsulates and delegates parsing to a child `DecisionProcess[StructuredIntent]`.

### How the Delegation Works
- Initialization (`__init__)`: `ProcessHumanIntent` instantiates a `DecisionProcess` configured with the `StructuredIntent` sub-state and registers the `ExtractIntent` and `ExtractIntentObject` operators (marking the latter as `terminal=True`).

- Guard Check (`can_perform`): Triggers only when a raw `Utterance` is present on the input container (`io.i.interaction`) and `state.human_intent` has not yet been resolved.

- Execution (`perform`): Reinitializes the sub-process and invokes it with the raw `utterance` as an argument. Once the sub-process terminates, its output (`StructuredIntent`) is saved directly back into the top-level `state.human_intent`.

In [ ]:
@dataclass
class HomeState:
    """Internal top-level state maintained by the cogent."""

    mode: Mode
    timer_expires_at: float = 0.0
    human_utterance: Utterance | None = None
    human_intent: StructuredIntent | None = None


class ProcessHumanIntent(Operator[HomeState]):
    """
    INIT: Instantiate DecisionProcess for language interpretation.
    WHEN: Raw utterance exists but structured human intent is not populated.
    THEN: Invoke child DecisionProcess to extract a fully resolved StructuredIntent.
    """

    def __init__(self, name: str, **kwargs: Any) -> None:
        super().__init__(name, **kwargs)
        self._human_utternace: Utterance | None = None
        self._dp: DecisionProcess[StructuredIntent] = DecisionProcess(StructuredIntent)
        self._dp.add_operator(ExtractIntent("extract_intent_class"))
        self._dp.add_operator(ExtractIntentDevice("extract_intent_device"))
        self._dp.add_operator(ExtractIntentMode("extract_intent_mode"))
        self._dp.add_termination_check(is_process_human_intent_terminal)

    def can_perform(self, state: HomeState, io: IOContainer) -> bool:
        self._human_utternace = cast(Utterance, io.i.interaction)
        return bool(self._human_utternace.phrase and not state.human_intent)

    def perform(self, state: HomeState, io: IOContainer) -> None:
        print("--> [Interpret Intent triggered]")
        self._dp.reinit()
        state.human_intent = self._dp(utterance=self._human_utternace)
        print(f"--> [Interpreted Intent] human expressed: {state.human_intent}")

## Step 5: Handle Unrecognized Intents (`ReportUnknownIntent`)

Residents will inevitably issue ambiguous, out-of-domain, or unsupported commands to any conversational agent (e.g., *"Make me a coffee"* or *"What's the weather today?"*).

Without explicit fallback handling, an agent might either:
- Stall silently: Leaving the user wondering if the system heard them.
- Execute erratic actions: Attempting hallucinated actions on real-world devices.

In `cognition`, we enforce a whitelist security model. When the sub-process `EnumClassifier` cannot map an utterance to a known `ResidentIntent` or `Devices`/`Mode`, it assigns `KState.UNKNOWN` to the sub-state fields. We then use a dedicated parent operator, `ReportUnknownIntent`, to safely catch these unmapped requests, notify the resident, and clear the interaction pipeline.

In [11]:
class ReportUnknownIntent(Operator[HomeState]):
    """
    WHEN: human_intent is unknown
    THEN: inform the user
    """

    def can_perform(self, state: HomeState, io: IOContainer) -> bool:
        if state.human_intent:
            return (
                state.human_intent.intent is KState.UNKNOWN
                or state.human_intent.object is KState.UNKNOWN
            )
        return False

    def perform(self, state: HomeState, io: IOContainer) -> None:
        io.o.interaction(Utterance(phrase=MyIntent.DONTKNOW))
        state.human_intent = None

## Step 6: Implement Action Execution Operators

Once `ProcessHumanIntent` stores the populated `StructuredIntent` in `state.human_intent`, parent-level execution operators (`ActuateDevice` and `SetMode`) evaluate the structured intent and trigger physical environment commands or mode changes deterministically.

In [ ]:
class ActuateLightDevice(Operator[HomeState]):
    """
    WHEN: human_intent.intent is ResidentIntent.TURN_ON/TURN_OFF and
    human_intent.object a LIGHT device
    THEN: Dispatch appropriate control signal
    """

    def __init__(self, name: str, **kwargs: Any) -> None:
        super().__init__(name, **kwargs)
        self._device: Devices | None = None
        self._signal: ControlSignal | None = None

    def can_perform(self, state: HomeState, io: IOContainer) -> bool:
        if (
            state.human_intent
            and isinstance(state.human_intent.object, Devices)
            and state.human_intent.object in LIGHTS
        ):
            self._device = cast(Devices, state.human_intent.object)
            if state.human_intent.intent is ResidentIntent.TURN_ON:
                self._signal = ControlSignal.TURN_ON
            elif state.human_intent.intent is ResidentIntent.TURN_OFF:
                self._signal = ControlSignal.TURN_OFF
            return True
        return False

    def perform(self, state: HomeState, io: IOContainer) -> None:
        io.o.devices(Action(device=self._device.value, signal=self._signal))
        io.o.interaction(Utterance(phrase=MyIntent.CONFIRMATION))
        print("--> [Action Triggered] Dispatching device signal.")
        state.human_intent = None


class SetMode(Operator[HomeState]):
    """
    WHEN: Intent targets a system Mode change.
    THEN: Update internal system mode.
    """

    def __init__(self, name: str, **kwargs: Any) -> None:
        super().__init__(name, **kwargs)
        self._mode: Mode | None = None

    def can_perform(self, state: HomeState, io: IOContainer) -> bool:
        if state.human_intent and (
            state.human_intent.intent is ResidentIntent.SET_MODE
            and isinstance(state.human_intent.object, Mode)
        ):
            self._mode = state.human_intent.object
            return True
        return False

    def perform(self, state: HomeState, io: IOContainer) -> None:
        state.mode = self._mode
        print(f"--> [Internal Action Triggered] Setting mode to {state.mode}")
        state.human_intent = None
        io.o.interaction(Utterance(phrase=MyIntent.CONFIRMATION))

## Step 7: Assemble the Hierarchical Cogent and Run

Finally, we connect our sensors and actuators to the `SmartHomeClient`, attach them to the `Cogent`, register top-level parent operators, and launch the cogent execution loop.

In [ ]:
# Connect to external Smart Home simulation client
client = SmartHomeClient()
try:
    client.connect()
    print("Successfully connected to Smart Home Simulation!")
except ConnectionError as exc:
    print(
        "Connection failed. Ensure the simulation app is running in another terminal:\n"
        "  python smart_home/smart_home.py --mode realistic\n"
        f"Details: {exc}"
    )


class HumanInteraction(Sensor[Utterance | None], Actuator[Utterance, None]):
    """Sensor and actuator interface for user chat interactions."""

    @property
    def name(self) -> str:
        return "interaction"

    def sense(self) -> Utterance | None:
        """Fetch the latest message from the human."""
        message = client.get_last_message()
        if message:
            return Utterance(phrase=message["text"])
        return Utterance(phrase=None)

    def actuate(self, param: Utterance) -> None:
        """Send a response to the human."""
        client.acknowledge_message()
        if param.phrase:
            client.send_message(param.phrase)


class SmartHomeDevices(Sensor[Observation], Actuator[Action, None]):
    """Sensor and actuator interface for physical home devices."""

    @property
    def name(self) -> str:
        return "devices"

    def sense(self) -> Observation:
        response = client.observe()
        return Observation.model_validate(response)

    def actuate(self, param: Action) -> None:
        client.actuate(str(param.device), param.signal)


state = HomeState(mode=Mode.MIDNIGHT)
dp = DecisionProcess[HomeState](lambda: state)
assistant: Cogent[DecisionProcess[HomeState]] = Cogent(decision_process=dp)
devices = SmartHomeDevices()
interaction = HumanInteraction()
assistant.add_sensor(devices).add_actuator(devices)
assistant.add_sensor(interaction).add_actuator(interaction)
assistant.dp.add_operator(ProcessHumanIntent("process_human_input"))
assistant.dp.add_operator(ReportUnknownIntent("report_unknown_intent", terminal=True))
assistant.dp.add_operator(SetMode("set_mode", terminal=True))
assistant.dp.add_operator(ActuateLightDevice("actuate_light_device", terminal=True))
run_cogent(assistant)

Try asking the cogent to "turn the light in the bedroom on" or "turn living room light off". What happens when you ask the cogent "run the dishwasher".

## Developer Exercise: Complete Implementation of Action Operators
If you asked the assistant to "run the dishwasher", you likely observed that while the sub-process correctly interpreted the resident's intent (`ResidentIntent.TURN_ON` with `Devices.DISHWASHER`), nothing happened in the simulation.

Our current `ActuateLightDevice` operator is explicitly constrained to evaluate device targets listed in the `LIGHTS` set. Non-lighting appliances such as the dishwasher, washer, dryer, and main door lock require distinct operational control signals (e.g., `ControlSignal.START` vs. `ControlSignal.TURN_ON`, or `ControlSignal.LOCK` vs. `ControlSignal.UNLOCK`).

To make the cogent fully functional across the entire home, your task is to design and register the missing execution operators to the main decision process.

### Tasks to Complete:
1. **Add sets representing `RUNNABLES` and `LOCKS`** Sets that contain  appliances like `Devices.DISHWASHER`, `Devices.WASHER`, and `Devices.DRYER`. And, `Devices.MAIN_DOOR`.
2. **Implement `Acutated*Device` Operators:**
   * Guard condition (`can_perform`): Validate that `state.human_intent` contains a recognized appliance target and a supported intent type before signaling execution readiness.
   * Action (`perform`): Dispatch an `Action` payload containing the target appliance and the corresponding start/stop signal to `io.o.devices`. Send a confirmation `Utterance` to `io.o.interaction` acknowledging the action. Reset transient pipeline variables (`state.human_intent = None` ) to mark the request as complete and prevent infinite execution loops.
3. **Register the Operator:** Add your new operator to `assistant.dp`.
4. **Test in the TUI:** Ask the assistant to "run the dishwasher". Remember that the TUI may be stuck in an odd state, clear that by restarting the TUI.

Check out the full implementation at [smart_home_assistant_v4.py](smart_home_assistant_v4.py).